# 02. Data Preprocessing & Feature Engineering
### Avoiding Data Leakage & Pipeline Architecture

Key principles:
1. **No Data Leakage**: Train/test split (80/20) happens before fitting any imputer, scaler, or encoder.
2. **Domain Feature Engineering**: Calculate property age, renovation flags, room totals, living space ratios, and distance to Seattle downtown.
3. **ColumnTransformer Integration**: Wrap transformations in scikit-learn standard pipelines.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

from src.config import RANDOM_STATE, TEST_SIZE, TARGET_COLUMN
from src.data_loader import load_raw_data, clean_metadata_columns
from src.feature_engineering import engineer_features
from src.preprocessing import build_preprocessor


## 1. Train/Test Split (80/20)


In [ ]:
df_raw = load_raw_data()
df_clean = clean_metadata_columns(df_raw)

train_df, test_df = train_test_split(
    df_clean, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=True
)
print(f"Train Shape: {train_df.shape} | Test Shape: {test_df.shape}")


## 2. Domain Feature Engineering


In [ ]:
train_engineered = engineer_features(train_df)
print("Engineered Columns:", [c for c in train_engineered.columns if c not in train_df.columns])
display(train_engineered[['house_age', 'is_renovated', 'total_rooms', 'distance_to_center_km']].head())


## 3. Fit ColumnTransformer on Training Data


In [ ]:
preprocessor = build_preprocessor()
X_train_proc = preprocessor.fit_transform(train_engineered.drop(columns=[TARGET_COLUMN]))
print("Transformed Feature Matrix Shape:", X_train_proc.shape)
